# P2C-64 v2.0 (Full PTM + 60-Ply Smooth Heads) — Google Colab 実行ノートブック（T4 / L4 / A100 自動対応）

Egaroucid 公開学習データ（`Egaroucid_Train_Data.zip`, 約2,551万局面）を用いて、**P2C-64 v2.0** の全5モデル（`RAW_NTUPLE_64`, `LINEAR_64`, `CAND_A_PTM_SCRELU4`, `CAND_B_PTM_3W3B`, `CAND_C_PTM_HYBRID`）を学習・監査・実戦対局するためのワンクリックノートブックです。

### v2.0 主な強化点
- **完全手番相対（PTM）骨格**：Phase 1 を1回だけ学習し、`raw_ntuple64.p2c` と `linear64.p2c` へ完全同一重みを出力（土台ハンデ厳密 0）。
- **60手番平滑化ヘッド ＋ 隣接ブロック平滑化**：SCReLUヘッド（`2.81 KiB`）および SCONE-D Macro-LUT（`180 KiB`, 全8コード解放）を60手番化し、6手境界段差と中盤 MPC $\sigma$ 悪化を解消。
- **既存キャッシュ完全互換**：Google Drive 上の既存 `cache_bin/{train.bin, val.bin}` を再加工なしでそのまま再利用し、出力は新規ディレクトリ `run_egaroucid_25m_v20/` へ保存します（v1.1 成果物は上書きしません）。

## Cell 1: GPU/VRAM 自動判定 ＆ Googleドライブマウント

In [ ]:
import os
import torch
from google.colab import drive

drive.mount("/content/drive")

RUN_NAME = "run_egaroucid_25m_v20"
DRIVE_DATA_DIR = "/content/drive/MyDrive/p2c64_data"
DRIVE_ZIP_PATH = f"{DRIVE_DATA_DIR}/Egaroucid_Train_Data.zip"
DRIVE_CACHE_DIR = f"{DRIVE_DATA_DIR}/cache_bin"
DRIVE_WORK_DIR = f"/content/drive/MyDrive/p2c64_runs/{RUN_NAME}"
LOCAL_DATA_DIR = "/content/data"
REPO_DIR = "/content/p2c64-othello"

os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
os.makedirs(DRIVE_WORK_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# Auto-configure batch size by GPU VRAM (T4 16GB vs L4 24GB vs A100 40GB+)
VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU     : {torch.cuda.get_device_name(0)} ({VRAM_GB:.1f} GiB VRAM)")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
else:
    print("Running on CPU runtime (suitable for Cell 7 pure-scalar matches & MPC sigma)")

P1_BATCH = 32768 if VRAM_GB >= 20.0 else 16384
P2_BATCH = 16384
VAL_BATCH = 65536 if VRAM_GB >= 20.0 else 32768
print(f"Auto Batch Sizes: P1_BATCH={P1_BATCH}, P2_BATCH={P2_BATCH}, VAL_BATCH={VAL_BATCH}")
!lscpu | grep -E "Model name|CPU\(s\):|L1d|L2|L3" | head -n 10

## Cell 2: リポジトリ `clone` / `pull` ＆ C++ツール・OpenMPブリッジビルド ＆ 監査

In [ ]:
%%bash
set -euo pipefail
REPO_URL="https://github.com/Taichi111009/p2c64-othello.git"
REPO_DIR="/content/p2c64-othello"

if [ -d "$REPO_DIR/.git" ]; then
    cd "$REPO_DIR" && git fetch origin && git pull --ff-only origin main
else
    git clone "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
make clean all test audit
ls -lh build/

## Cell 3: スマートデータ準備（既存 Drive `train.bin` / `val.bin` キャッシュをそのまま再利用）

In [ ]:
import os
import shutil
import subprocess

cache_tr = f"{DRIVE_CACHE_DIR}/train.bin"
cache_va = f"{DRIVE_CACHE_DIR}/val.bin"
local_tr = f"{LOCAL_DATA_DIR}/train.bin"
local_va = f"{LOCAL_DATA_DIR}/val.bin"

if os.path.exists(local_tr) and os.path.exists(local_va):
    print("[OK] Local SSD binary cache already present:", local_tr, local_va)
elif os.path.exists(cache_tr) and os.path.exists(cache_va):
    print("[Cache Hit] Reusing existing v1.1/v2.0 compatible train.bin / val.bin from Google Drive -> Local SSD...")
    shutil.copy2(cache_tr, local_tr)
    shutil.copy2(cache_va, local_va)
    print("[OK] Copied to local SSD!")
else:
    assert os.path.exists(DRIVE_ZIP_PATH), f"Input ZIP not found: {DRIVE_ZIP_PATH}"
    local_zip = "/content/Egaroucid_Train_Data.zip"
    unzip_dir = "/content/egaroucid_raw"
    shutil.copy2(DRIVE_ZIP_PATH, local_zip)
    os.makedirs(unzip_dir, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", unzip_dir], check=True)
    subprocess.run([
        f"{REPO_DIR}/build/preprocess_egaroucid",
        "--input-dir", unzip_dir,
        "--out-train", local_tr,
        "--out-val", local_va,
        "--val-pct", "5",
    ], check=True)
    shutil.copy2(local_tr, cache_tr)
    shutil.copy2(local_va, cache_va)
    os.remove(local_zip)
    shutil.rmtree(unzip_dir, ignore_errors=True)
    print("[OK] Data preparation & Drive caching complete!")

!ls -lh /content/data/

## Cell 4: Phase 1 学習（1回のみ学習 → `RAW_NTUPLE_64` ＆ `LINEAR_64` へ完全同一重みを自動保存）

In [ ]:
!python3 {REPO_DIR}/python/train_p2c64.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --work-dir {DRIVE_WORK_DIR} \
    --phases phase1 \
    --resume \
    --batch-size {P1_BATCH} \
    --val-batch-size {VAL_BATCH} \
    --p1-epochs 6 \
    --p1-lr 0.05 \
    --smooth-lambda 1e-4 \
    --huber-beta 4.0

## Cell 5: Phase 2 学習（v2.0 候補A・B・C、60手番平滑化ヘッド ＆ 隣接ブロック平滑化）＋ Val RMSE サマリー

In [ ]:
!python3 {REPO_DIR}/python/train_p2c64.py \
    --train-bin {LOCAL_DATA_DIR}/train.bin \
    --val-bin {LOCAL_DATA_DIR}/val.bin \
    --work-dir {DRIVE_WORK_DIR} \
    --phases phase2 \
    --resume \
    --batch-size {P2_BATCH} \
    --val-batch-size {VAL_BATCH} \
    --head-smooth-lambda 1e-3 --blk-smooth-lambda 5e-4 \
    --a-epochs 4 --a-qat-epochs 2 --a-lr 0.01 --mu-init 0.5 \
    --b-soft-epochs 4 --b-lr 0.02 --tau-start 1.5 --tau-end 0.15 --eb-lambda 200.0 --eb-ply-lambda 100.0 --eb-amax 4.0 --em-iters 2 \
    --c-screlu-epochs 3 --c-qat-epochs 1 --c-soft-epochs 3 --c-em-iters 2

print("\nSaved v2.0 artifacts on Google Drive:")
!ls -lh {DRIVE_WORK_DIR}/*.p2c {DRIVE_WORK_DIR}/metrics.*

## Cell 6: C++/Python ビット完全一致・`RAW == LINEAR` 厳密一致・16対称（$D_4 \times \mathbb{Z}_2$）監査

In [ ]:
!python3 {REPO_DIR}/python/test_bit_exact_io.py \
    --data {LOCAL_DATA_DIR}/val.bin \
    --num-pos 1000 \
    --work-dir {DRIVE_WORK_DIR} \
    --bench-bridge

## Cell 7: （GPU切断後のCPUランタイム推奨）純スカラー実戦1,000局ペア対局 ＆ MPC $\sigma$ 計測

CPUランタイムへ切り替えた後、Cell 1〜2 を再実行してから本セルを実行してください。`RAW_NTUPLE_64` に対する `LINEAR_64` / `CAND_A` / `CAND_B` / `CAND_C` の `nodes=20000`（1,000局）および `time=20ms`（1,000局）対局と、`d2→d6`・`d4→d8` MPC $\sigma$ を計測して Google Drive へ CSV 保存します。

In [ ]:
import os
import subprocess

MATCH_BIN = f"{REPO_DIR}/build/match_engine"
MPC_BIN = f"{REPO_DIR}/build/measure_mpc_sigma"
RAW_P2C = f"{DRIVE_WORK_DIR}/raw_ntuple64.p2c"
LIN_P2C = f"{DRIVE_WORK_DIR}/linear64.p2c"
CANDA_P2C = f"{DRIVE_WORK_DIR}/cand_a_ptm_screlu4.p2c"
CANDB_P2C = f"{DRIVE_WORK_DIR}/cand_b_ptm_3w3b.p2c"
CANDC_P2C = f"{DRIVE_WORK_DIR}/cand_c_ptm_hybrid.p2c"

val_bin = f"{LOCAL_DATA_DIR}/val.bin" if os.path.exists(f"{LOCAL_DATA_DIR}/val.bin") else f"{DRIVE_CACHE_DIR}/val.bin"

# 1) MPC sigma measurement (depth 2 -> 6 and depth 4 -> 8 on 2,000 unseen positions)
for sh, dp in [(2, 6), (4, 8)]:
    print(f"\n=== P2C-64 v2.0 MPC Sigma: depth {sh} -> {dp} ===")
    subprocess.run([
        MPC_BIN,
        "--data", val_bin,
        "--num-pos", "2000",
        "--shallow", str(sh),
        "--deep", str(dp),
        "--tt-mb", "64",
        "--csv", f"{DRIVE_WORK_DIR}/mpc_sigma_d{sh}_d{dp}.csv",
        "--models", RAW_P2C, LIN_P2C, CANDA_P2C, CANDB_P2C, CANDC_P2C,
    ], check=True)

# 2) 1,000-game paired matches (500 pairs): Equal-Nodes & Equal-Time vs RAW_NTUPLE_64
match_csv = f"{DRIVE_WORK_DIR}/match_results_v20.csv"
for cand_path in [LIN_P2C, CANDA_P2C, CANDB_P2C, CANDC_P2C]:
    for mode, limit in [("nodes", "20000"), ("time", "20")]:
        print(f"\n=== 1,000-Game Match ({mode}={limit}): {os.path.basename(cand_path)} vs RAW_NTUPLE_64 ===")
        subprocess.run([
            MATCH_BIN,
            "--p1", cand_path,
            "--p2", RAW_P2C,
            "--mode", mode,
            "--limit", limit,
            "--pairs", "500",
            "--opening-plies", "8",
            "--endgame-empties", "10",
            "--tt-mb", "128",
            "--seed", "20261125",
            "--csv", match_csv,
        ], check=True)

print(f"\nAll v2.0 match & MPC results saved to {DRIVE_WORK_DIR}")